# Phase 4: Evaluation & Quantization

This notebook runs benchmark evaluations on the final model and handles quantization to GGUF formats (`Q3_K_M`, `Q4_K_M`, `Q5_K_M`) for local deployment on your laptop via Ollama / LM Studio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install lm-evaluation-harness huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    pass

## 1. Automated Benchmarking

We use the `lm-evaluation-harness` to run evaluations directly on the FP16 merged weights.

In [ ]:
model_path = "/content/drive/MyDrive/btech-ai-tutor/models/btech_ai_tutor_7b_final"
!lm_eval --model hf \
    --model_args pretrained={model_path},dtype=float16 \
    --tasks mmlu_computer_science,mmlu_college_computer_science,gsm8k \
    --batch_size 4 \
    --output_path /content/drive/MyDrive/btech-ai-tutor/evaluation_results.json

## 2. Quantization to GGUF

Unsloth supports direct quantization to GGUF format which integrates llama.cpp conversion. We will export 3 quantizations:
1. **q3_k_m**: Extremely lightweight, fits in 4GB VRAM with minimal overhead.
2. **q4_k_m**: Highly recommended balance, fast inference with strong accuracy.
3. **q5_k_m**: Best accuracy, run on CPU with 16GB system RAM if GPU bottlenecks.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/btech-ai-tutor/models/btech_ai_tutor_7b_final",
    max_seq_length = 4096,
    dtype = torch.float16,
    load_in_4bit = False # Loading in full float16 is better before quantization
)

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/btech-ai-tutor/models/gguf_q3",
    tokenizer,
    quantization_method = "q3_k_m"
)

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/btech-ai-tutor/models/gguf_q4",
    tokenizer,
    quantization_method = "q4_k_m"
)

In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/btech-ai-tutor/models/gguf_q5",
    tokenizer,
    quantization_method = "q5_k_m"
)

## 3. Upload to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

repo_name = "YOUR_HF_USERNAME/btech-ai-tutor-7b-GGUF"

try:
    api.create_repo(repo_id=repo_name, repo_type="model")
except Exception as e:
    print(f"Repo may already exist: {e}")

api.upload_folder(
    folder_path="/content/drive/MyDrive/btech-ai-tutor/models/gguf_q3",
    repo_id=repo_name,
    path_in_repo="q3_k_m"
)

api.upload_folder(
    folder_path="/content/drive/MyDrive/btech-ai-tutor/models/gguf_q4",
    repo_id=repo_name,
    path_in_repo="q4_k_m"
)

api.upload_folder(
    folder_path="/content/drive/MyDrive/btech-ai-tutor/models/gguf_q5",
    repo_id=repo_name,
    path_in_repo="q5_k_m"
)

print("All GGUF quantized models uploaded successfully!")